In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = Path(
    "/Users/nestoralaminospelegrina/Proyectos/portfolio_dashboard/"
    "data/raw/myinvestor/fondos/"
    "plusvalias_minusvalias_E000ZYRH0Q7_20261022.csv"
)

df = pd.read_csv(
    CSV_PATH,
    sep=";",
    decimal=",",
    parse_dates=["Fecha fiscal"],
    date_format="%d/%m/%Y",
)

df = df.sort_values("Fecha fiscal").reset_index(drop=True)

df.head()

In [ ]:
columnas_numericas = [
    "Inversión",
    "Valor de mercado",
    "Resultado fiscal",
]

df[columnas_numericas] = df[columnas_numericas].apply(
    pd.to_numeric, errors="raise"
)

assert df["Fecha fiscal"].notna().all()
assert df[columnas_numericas].notna().all().all()

# Comprobar que el resultado fiscal coincide con la diferencia.
df["Resultado calculado"] = (
    df["Valor de mercado"] - df["Inversión"]
)

df["Diferencia de control"] = (
    df["Resultado fiscal"] - df["Resultado calculado"]
)

display(df.round(2))

inversion_total = df["Inversión"].sum()
valor_actual = df["Valor de mercado"].sum()
resultado_total = df["Resultado fiscal"].sum()

rentabilidad_simple = (
    valor_actual / inversion_total - 1
) * 100

print(f"Inversión declarada: {inversion_total:,.2f} €")
print(f"Valor de mercado:    {valor_actual:,.2f} €")
print(f"Resultado fiscal:    {resultado_total:,.2f} €")
print(f"Rentabilidad simple: {rentabilidad_simple:.2f} %")
print(
    "Máximo descuadre de control:",
    df["Diferencia de control"].abs().max().round(4),
)

In [ ]:
import yfinance as yf

tickers = {
    "S&P 500": "^GSPC",
    "Nasdaq Composite": "^IXIC",
}

fecha_inicio = df["Fecha fiscal"].min().strftime("%Y-%m-%d")

# La fecha final de yfinance es exclusiva.
precios = yf.download(
    list(tickers.values()),
    start=fecha_inicio,
    interval="1d",
    auto_adjust=False,
    progress=False,
)

if precios.empty:
    raise ValueError("No se han podido descargar precios históricos.")

# Para índices, usamos el cierre publicado.
if isinstance(precios.columns, pd.MultiIndex):
    cierres = precios["Close"].copy()
else:
    cierres = precios[["Close"]].copy()
    cierres.columns = list(tickers.values())

cierres = cierres.rename(
    columns={v: k for k, v in tickers.items()}
)

cierres.index = pd.to_datetime(cierres.index).tz_localize(None)
cierres = cierres.sort_index().dropna(how="all")

cierres.tail()

In [ ]:
# Usamos el calendario de sesiones bursátiles de cada índice.
# Para cada fecha de referencia, tomamos el último cierre disponible
# en esa fecha o antes de ella.

resultados = []

for nombre_indice in cierres.columns:
    serie = cierres[nombre_indice].dropna()

    for _, fila in df.iterrows():
        fecha = fila["Fecha fiscal"]
        importe = fila["Inversión"]

        precios_previos = serie.loc[serie.index <= fecha]

        if precios_previos.empty:
            raise ValueError(
                f"No hay precio de {nombre_indice} para {fecha.date()}"
            )

        precio_compra = precios_previos.iloc[-1]
        precio_final = serie.iloc[-1]

        valor_simulado = importe * precio_final / precio_compra

        resultados.append({
            "Índice": nombre_indice,
            "Fecha": fecha,
            "Inversión": importe,
            "Precio de referencia": precio_compra,
            "Precio final": precio_final,
            "Valor simulado": valor_simulado,
        })

simulacion = pd.DataFrame(resultados)

resumen_indices = (
    simulacion.groupby("Índice")
    .agg(
        Capital_invertido=("Inversión", "sum"),
        Valor_final_simulado=("Valor simulado", "sum"),
    )
)

resumen_indices["Resultado_simulado"] = (
    resumen_indices["Valor_final_simulado"]
    - resumen_indices["Capital_invertido"]
)

resumen_indices["Rentabilidad_pct"] = (
    resumen_indices["Valor_final_simulado"]
    / resumen_indices["Capital_invertido"] - 1
) * 100

display(resumen_indices.round(2))

In [ ]:
comparativa = pd.concat([
    pd.DataFrame({
        "Valor final (€)": [valor_actual],
    }, index=["Mi fondo (según CSV)"]),
    resumen_indices[["Valor_final_simulado"]].rename(
        columns={"Valor_final_simulado": "Valor final (€)"}
    ),
])

display(comparativa.round(2))

comparativa["Valor final (€)"].plot(
    kind="bar",
    figsize=(10, 5),
    title="Valor final: fondo frente a índices de referencia",
    ylabel="Euros",
    xlabel="Alternativa",
    grid=True,
)

plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px

# 'cierres' debe contener los precios históricos de cada índice.
# Cada columna es una serie y el índice son las fechas.

precios_normalizados = cierres.apply(
    lambda serie: serie / serie.dropna().iloc[0] * 100
)

fig = px.line(
    precios_normalizados,
    x=precios_normalizados.index,
    y=precios_normalizados.columns,
    title="Evolución normalizada de los índices",
    labels={
        "value": "Índice (base 100)",
        "variable": "Referencia",
        "Date": "Fecha",
    },
)

fig.update_layout(
    hovermode="x unified",
    yaxis_title="Base 100",
    xaxis_title="Fecha",
    legend_title="Inversión",
)

fig.show(renderer="browser")

In [ ]:
import sys
import nbformat
import plotly

print("Python:", sys.executable)
print("nbformat:", nbformat.__version__)
print("Plotly:", plotly.__version__)